# Notebook to define and evaluate a baseline Siamese UNet

Note that UNet baseline uses early fusion whereas Siamese uses late fusion via feature differencing. In other words, these two architectures differ in how input data is fused, and evaluation of these architectures must consider these differences.

In [1]:
import os
import sys

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("CWD:", os.getcwd())
print("Project root added:", project_root)

CWD: c:\Dev\Damage_Assessment_on_xBD\Model_Architectures Notebooks
Project root added: c:\Dev\Damage_Assessment_on_xBD


In [2]:
import torch
import torch.optim as optim

from src.dataloader import get_loaders
from src.eval import test_evaluation
from src.train import ComboLoss, run_training, plot_train_history
from  src.model_siamese import SiameseUNet
from src.visualizations import visualize_predictions

In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print("Using CPU")

Using GPU: NVIDIA GeForce RTX 4070


In [4]:
# Config Hyperparameters (Adjust)
NUM_CLASSES = 5
BATCH_SIZE = 8
LR = 1e-4
NUM_EPOCHS = 50
PATIENCE = 12
BASE_FEATURES = 32
SAVE_PATH = "best_siamese_unet_combo.pth"

In [5]:
# Create Data Splits
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from Preprocessing.xBD_splits import create_splits
img_dir = r"..\Preprocessing\tiles\images"
mask_dir = r"..\Preprocessing\tiles\masks"
train_files, val_files, test_files = create_splits(img_dir, mask_dir)
print(f"Train: {len(train_files)}  Val: {len(val_files)}  Test: {len(test_files)}")

Train: 9820
Val: 1960
Test: 1912
Train: 9820  Val: 1960  Test: 1912


In [6]:
train_loader, val_loader, test_loader = get_loaders(
    train_files, val_files, test_files, img_dir, mask_dir,
    batch_size=BATCH_SIZE, num_workers = 4, pin_memory = True, siamese=True, augment_train=True)

In [7]:
# Sanity check
pre, post, masks = next(iter(train_loader))

print(f"Pre image:  {pre.shape}")    # [B, 3, H, W]
print(f"Post image: {post.shape}")   # [B, 3, H, W]
print(f"Masks:      {masks.shape}")  # [B, H, W]
print(f"Mask range: {masks.min()} – {masks.max()}")
print(f"Device:     {pre.device}")

Pre image:  torch.Size([8, 3, 512, 512])
Post image: torch.Size([8, 3, 512, 512])
Masks:      torch.Size([8, 512, 512])
Mask range: 0 – 4
Device:     cpu


In [8]:
# Loss — ComboLoss (CE + Tversky)
class_counts = torch.tensor([
    3258249517, 287893139, 15076019, 19138232, 8918741
], dtype=torch.float32)

freqs = class_counts / class_counts.sum()
weights = 1.0 / torch.sqrt(freqs)
weights = weights / weights.mean()
class_weights = weights.to(device)

In [9]:
# Model
model = SiameseUNet(
    num_classes=NUM_CLASSES,
    base_features=BASE_FEATURES,
    in_channels=3 # only include 3 channels for Siamese
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(next(model.parameters()).device)

Model parameters: 10,378,469
cuda:0


In [10]:
# Loss, Optimizer, LR Scheduler (Define Here)
# Heavily penalize FN damage pixels with heavily weighted tversky loss and tversky_alpha
criterion = ComboLoss(class_weights=class_weights, ce_weight=0.3,
                      tversky_weight=0.7, tversky_alpha=0.8,
                      tversky_beta=0.2, num_classes=NUM_CLASSES).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)

In [ ]:
history = run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    num_classes=NUM_CLASSES,
    save_path=SAVE_PATH, verbose = True,
    siamese = True)


c:\Dev\Damage_Assessment_on_xBD\src\eval.py:65: RuntimeWarning: invalid value encountered in divide
  precision = np.where(denom_prec > 0, TP / denom_prec, np.nan)



Epoch 1/50  (lr=1.00e-04)
  Train Loss: 0.7712  |  Val Loss: 0.7734
  Train mIoU: 0.0206  |  Val mIoU: 0.0070
  Train Acc:  0.0475  |  Val Acc:  0.0203
  Val Per-Class:
    Class                IoU     Prec   Recall
    -----------------------------------------
    No Damage         0.0000      nan   0.0000
    Minor             0.0262   0.0272   0.4149
    Major             0.0000   0.0000   0.0157
    Destroyed         0.0013   0.0013   0.2624
    Unclassified      0.0076   0.0076   0.6442
  *** Saved new best model (mIoU=0.0070) ***


  *** Saved new best model (mIoU=0.0117) ***



Epoch 5/50  (lr=2.53e-06)
  Train Loss: 0.8001  |  Val Loss: 0.8069
  Train mIoU: 0.0247  |  Val mIoU: 0.0112
  Train Acc:  0.0584  |  Val Acc:  0.0320
  Val Per-Class:
    Class                IoU     Prec   Recall
    -----------------------------------------
    No Damage         0.0000      nan   0.0000
    Minor             0.0432   0.0441   0.6691
    Major             0.0000   0.0000   0.0202
    Destroyed         0.0030   0.0032   0.0521
    Unclassified      0.0097   0.0098   0.6548


c:\Dev\Damage_Assessment_on_xBD\src\eval.py:78: RuntimeWarning: invalid value encountered in divide
  2 * precision * recall / (precision + recall),


  *** Saved new best model (mIoU=0.0126) ***



Epoch 10/50  (lr=9.85e-05)
  Train Loss: 0.8364  |  Val Loss: 0.8332
  Train mIoU: 0.0207  |  Val mIoU: 0.0126
  Train Acc:  0.0572  |  Val Acc:  0.0407
  Val Per-Class:
    Class                IoU     Prec   Recall
    -----------------------------------------
    No Damage         0.0000      nan   0.0000
    Minor             0.0432   0.0435   0.8675
    Major             0.0001   0.0001   0.0153
    Destroyed         0.0041   0.0043   0.0832
    Unclassified      0.0158   0.0161   0.4038


Train:  46%|████▌     | 564/1228 [02:30<02:50,  3.89it/s]

In [ ]:
plot_train_history(history)

In [ ]:
test_loss, test_metrics = test_evaluation(model, test_loader, criterion, device, SAVE_PATH, NUM_CLASSES, siamese=True)

In [ ]:
visualize_predictions(model=model, test_loader=test_loader, device=device,
    num_samples=5, title="Siamese U-Net Predictions")